<style>
    body {
        margin: 40px 60px !important;
        padding: 20px !important;
        max-width: 1200px !important;
        margin-left: auto !important;
        margin-right: auto !important;
        background-color: #fdfdfd !important;
    }
    
    .container {
        padding: 20px 40px !important;
    }
    
    /* Dla wykresów - żeby nie wychodziły poza margines */
    img, svg, .output_png, .output_svg {
        max-width: 100% !important;
        height: auto !important;
    }
    
    /* Dla tabel */
    table {
        display: block !important;
        overflow-x: auto !important;
        white-space: nowrap !important;
    }
    
    /* Dla kodu */
    .input_area, .output_area {
        max-width: 100% !important;
        overflow-x: auto !important;
    }
    
    /* Dla notebooka */
    .notebook-container {
        padding: 30px 50px !important;
    }
</style>

<h2 id="czyszczenie">1. Czyszczenie danych</h2>

W tym notatniku przeprowadzimy pełny proces czyszczenia danych ze zbioru `data/sklep_rowerowy.csv`. Zidentyfikujemy brakujące wartości, sprawdzimy występowanie duplikatów, przeanalizujemy typy danych, zdiagnozujemy wartości odstające (outliery) oraz dokonamy imputacji braków za pomocą odpowiednich narzędzi.

### Wczytanie bibliotek
Rozpoczynamy od importu bibliotek `pandas` oraz `numpy`, które potrzebujemy do uporządkowania i czyszczenia danych.

In [ ]:
import pandas as pd
import numpy as np

### Wczytanie danych
Wczytujemy plik CSV o nazwie `data/sklep_rowerowy.csv` do ramki danych biblioteki pandas (`DataFrame`) i wyświetlamy podgląd pierwszych pięciu wierszy oraz rozmiar zbioru.

In [ ]:
df = pd.read_csv('data/sklep_rowerowy.csv')
print(f"Rozmiar zbioru: {df.shape[0]} wierszy, {df.shape[1]} kolumn.\n")
df.head(5)

Rozmiar zbioru: 1000 wierszy, 13 kolumn.



,ID,Marital Status,Gender,Income,Children,Education,Occupation,Home Owner,Cars,Commute Distance,Region,Age,Purchased Bike
0,12496,Married,Female,40000.0,1.0,Bachelors,Skilled Manual,Yes,0.0,0-1 Miles,Europe,42.0,No
1,24107,Married,Male,30000.0,3.0,Partial College,Clerical,Yes,1.0,0-1 Miles,Europe,43.0,No
2,14177,Married,Male,80000.0,5.0,Partial College,Professional,No,2.0,2-5 Miles,Europe,60.0,No
3,24381,Single,NaN,70000.0,0.0,Bachelors,Professional,Yes,1.0,5-10 Miles,Pacific,41.0,Yes
4,25597,Single,Male,30000.0,0.0,Bachelors,Clerical,No,0.0,0-1 Miles,Europe,36.0,Yes


Następnie, możemy sprawdzić informacje o naszych danych.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                1000 non-null   int64  
 1   Marital Status    993 non-null    object 
 2   Gender            989 non-null    object 
 3   Income            994 non-null    float64
 4   Children          992 non-null    float64
 5   Education         1000 non-null   object 
 6   Occupation        1000 non-null   object 
 7   Home Owner        996 non-null    object 
 8   Cars              991 non-null    float64
 9   Commute Distance  1000 non-null   object 
 10  Region            1000 non-null   object 
 11  Age               992 non-null    float64
 12  Purchased Bike    1000 non-null   object 
dtypes: float64(4), int64(1), object(8)
memory usage: 101.7+ KB


### Konwersja typów na całkowite (int) oraz tekstowe (string)
Po wczytaniu możemy zmienić kolumny na odpowiednie typy docelowe. Kolumny numeryczne (`Income`, `Children`, `Cars`, `Age`) konwertujemy na typ całkowitoliczbowy (`Int64`), a kolumnę `ID` oraz pozostałe kolumny tekstowe na typ tekstowy (`string`). Robimy to, bo typy te w pandas obsługują wartości brakujące (NaN).

In [ ]:
df['ID'] = df['ID'].astype('string')

cols_to_int = ['Income', 'Children', 'Cars', 'Age']
for col in cols_to_int:
    df[col] = df[col].astype('Int64')

categorical_cols = ['Marital Status', 'Gender', 'Education', 'Occupation', 'Home Owner', 'Commute Distance', 'Region', 'Purchased Bike']
for col in categorical_cols:
    df[col] = df[col].astype('string')

print("--- Typy danych kolumn po konwersji ---")
print(df.dtypes.to_string())
df.head()

--- Typy danych kolumn po konwersji ---
ID                  string[python]
Marital Status      string[python]
Gender              string[python]
Income                       Int64
Children                     Int64
Education           string[python]
Occupation          string[python]
Home Owner          string[python]
Cars                         Int64
Commute Distance    string[python]
Region              string[python]
Age                          Int64
Purchased Bike      string[python]


,ID,Marital Status,Gender,Income,Children,Education,Occupation,Home Owner,Cars,Commute Distance,Region,Age,Purchased Bike
0,12496,Married,Female,40000,1,Bachelors,Skilled Manual,Yes,0,0-1 Miles,Europe,42,No
1,24107,Married,Male,30000,3,Partial College,Clerical,Yes,1,0-1 Miles,Europe,43,No
2,14177,Married,Male,80000,5,Partial College,Professional,No,2,2-5 Miles,Europe,60,No
3,24381,Single,<NA>,70000,0,Bachelors,Professional,Yes,1,5-10 Miles,Pacific,41,Yes
4,25597,Single,Male,30000,0,Bachelors,Clerical,No,0,0-1 Miles,Europe,36,Yes


### Puste komórki
Sprawdzamy, ile brakujących wartości znajduje się w każdej kolumnie, aby dowiedzieć się, które kolumny wymagają imputacji.

In [ ]:
missing_values = df.isnull().sum()
print("Liczba braków w poszczególnych kolumnach:")
print(missing_values)

Liczba braków w poszczególnych kolumnach:
ID                   0
Marital Status       7
Gender              11
Income               6
Children             8
Education            0
Occupation           0
Home Owner           4
Cars                 9
Commute Distance     0
Region               0
Age                  8
Purchased Bike       0
dtype: int64


### Duplikaty
Sprawdzamy, czy w naszym zbiorze występują zduplikowane wiersze oraz czy identyfikatory klientów (`ID`) są unikalne.

In [ ]:
total_duplicates = df.duplicated().sum()
id_duplicates = df['ID'].duplicated().sum()
print(f"Liczba całkowicie zduplikowanych wierszy: {total_duplicates}")
print(f"Liczba zduplikowanych identyfikatorów ID: {id_duplicates}")

Liczba całkowicie zduplikowanych wierszy: 0
Liczba zduplikowanych identyfikatorów ID: 0


### Sprawdzenie unikalnych wartości i jeszcze raz (na wszelki wypadek) typów kolumn
Przed modyfikacją danych sprawdzamy typy danych poszczególnych kolumn oraz unikalne wartości dla kolumn tekstowych (kategorycznych). Pozwoli to zlokalizować ewentualne ukryte problemy (np. spacje w tekście czy inne nietypowe wartości).

In [ ]:
print("--- Typy danych kolumn ---")
print(df.dtypes)

print("\n--- Unikalne wartości w kolumnach kategorycznych ---")
categorical_cols = ['Marital Status', 'Gender', 'Education', 'Occupation', 'Home Owner', 'Commute Distance', 'Region', 'Purchased Bike']
for col in categorical_cols:
    print(f"\nKolumna '{col}': {df[col].unique()}")

--- Typy danych kolumn ---
ID                  string[python]
Marital Status      string[python]
Gender              string[python]
Income                       Int64
Children                     Int64
Education           string[python]
Occupation          string[python]
Home Owner          string[python]
Cars                         Int64
Commute Distance    string[python]
Region              string[python]
Age                          Int64
Purchased Bike      string[python]
dtype: object

--- Unikalne wartości w kolumnach kategorycznych ---

Kolumna 'Marital Status': <StringArray>
['Married', 'Single', <NA>]
Length: 3, dtype: string

Kolumna 'Gender': <StringArray>
['Female', 'Male', <NA>]
Length: 3, dtype: string

Kolumna 'Education': <StringArray>
[          'Bachelors',     'Partial College',         'High School',
 'Partial High School',     'Graduate Degree']
Length: 5, dtype: string

Kolumna 'Occupation': <StringArray>
['Skilled Manual', 'Clerical', 'Professional', 'Manual', '

### Diagnostyka wartości odstających
Sprawdzamy, czy w kolumnach numerycznych (`Income`, `Children`, `Cars`, `Age`) występują wartości odstające (outliery). Użyjemy do tego metody IQR (rozstęp ćwiartkowy), wyświetlając medianę, logiczne granice zakresu typowego (zabezpieczone przed wartościami ujemnymi) oraz 5 najbardziej odstających wartości.

In [ ]:
numerical_cols = ['Income', 'Children', 'Cars', 'Age']
print("--- Diagnostyka wartości odstających (metoda IQR) ---")
for col in numerical_cols:
    # Obliczamy kwartyle i rozstęp ćwiartkowy (IQR) dla niepustych wartości
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Określamy teoretyczne granice wartości odstających
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Ponieważ dochód, dzieci, samochody nie mogą być ujemne, dolną granicę dla celów prezentacyjnych ograniczamy do 0
    logical_lower_bound = max(0, lower_bound)
    
    # Filtrujemy outliery z wykorzystaniem matematycznych granic
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].copy()
    median_val = df[col].median()
    
    print(f"\nKolumna '{col}':")
    print(f"  Mediana: {median_val}")
    print(f"  Zakres typowy (IQR): [{logical_lower_bound}, {upper_bound}]")
    print(f"  Liczba zidentyfikowanych outlierów: {len(outliers)}")
    
    if len(outliers) > 0:
        # Obliczamy odległość od mediany i sortujemy malejąco
        outliers['distance'] = (outliers[col] - median_val).abs()
        top_outliers = outliers.sort_values(by='distance', ascending=False).head(5)
        print("  5 najbardziej odstających wartości:")
        for idx, row in top_outliers.iterrows():
            val = row[col]
            print(f"    Wartość: {val} (odległość od mediany: {abs(val - median_val)})")

--- Diagnostyka wartości odstających (metoda IQR) ---

Kolumna 'Income':
  Mediana: 60000.0
  Zakres typowy (IQR): [0, 130000.0]
  Liczba zidentyfikowanych outlierów: 10
  5 najbardziej odstających wartości:
    Wartość: 170000 (odległość od mediany: 110000.0)
    Wartość: 170000 (odległość od mediany: 110000.0)
    Wartość: 170000 (odległość od mediany: 110000.0)
    Wartość: 160000 (odległość od mediany: 100000.0)
    Wartość: 160000 (odległość od mediany: 100000.0)

Kolumna 'Children':
  Mediana: 2.0
  Zakres typowy (IQR): [0, 7.5]
  Liczba zidentyfikowanych outlierów: 0

Kolumna 'Cars':
  Mediana: 1.0
  Zakres typowy (IQR): [0, 3.5]
  Liczba zidentyfikowanych outlierów: 59
  5 najbardziej odstających wartości:
    Wartość: 4 (odległość od mediany: 3.0)
    Wartość: 4 (odległość od mediany: 3.0)
    Wartość: 4 (odległość od mediany: 3.0)
    Wartość: 4 (odległość od mediany: 3.0)
    Wartość: 4 (odległość od mediany: 3.0)

Kolumna 'Age':
  Mediana: 43.0
  Zakres typowy (IQR): [9.5, 

#### Co robimy z wartościami odstającymi?
Po analizie wyników identyfikacji wartości odstających podejmujemy następujące decyzje:
1. **Wysokie zarobki (Income):** Kilka osób posiada dochód powyżej górnej granicy (np. 160 000 - 170 000 USD). Nie są to jednak błędy w danych, lecz naturalna grupa zamożniejszych klientów. **Decyzja: pozostawiamy te dane bez zmian.**
2. **Wiek (Age):** Maksymalny wiek w zbiorze to 89 lat, co również jest realną wartością i nie stanowi błędu systemowego. **Decyzja: pozostawiamy te dane bez zmian.**
3. **Liczba dzieci (Children) i samochodów (Cars):** Kolumny te nie wykazują anomalii matematycznych ani wartości wykraczających poza logiczny zakres.
4. **Podsumowanie:** Wszystkie zidentyfikowane matematycznie outliery reprezentują rzeczywiste i poprawne scenariusze życiowe klientów. Ich usunięcie zniekształciłoby późniejszą analizę segmentacji klientów. Dlatego **decydujemy się pozostawić je w zbiorze danych**, a przy analizie statystycznej i wizualizacji (np. wykresy pudełkowe) będziemy pamiętać o ich obecności.

### Podstawowa obróbka danych – usuwanie spacji
Stosujemy metodę `.str.strip()` do wszystkich kolumn tekstowych (typu `string`), aby pozbyć się zbędnych spacji na początku i na końcu ciągów znaków. Metoda ta automatycznie ignoruje wartości brakujące.

In [ ]:
for col in df.select_dtypes(include=['string']).columns:
    df[col] = df[col].str.strip()
print("Usunięto zbędne spacje z kolumn tekstowych.")

Usunięto zbędne spacje z kolumn tekstowych.


### Imputacja brakujących wartości
Przechodzimy do uzupełnienia braków danych:
1. Kolumny jakościowe (kategoryczne: `Marital Status`, `Gender`, `Home Owner`) uzupełniamy **modą**.
2. Kolumny ilościowe (numeryczne: `Income`, `Children`, `Cars`, `Age`) uzupełniamy **medianą**, co chroni nas przed wpływem ewentualnych wartości odstających.

In [ ]:
df_before = df.copy()

# Imputacja kolumn kategorycznych za pomocą mody
cat_to_impute = ['Marital Status', 'Gender', 'Home Owner']
for col in cat_to_impute:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Kolumna '{col}': uzupełniono modą -> '{mode_val}'")

# Imputacja kolumn numerycznych za pomocą mediany
num_to_impute = ['Income', 'Children', 'Cars', 'Age']
for col in num_to_impute:
    
    median_val = int(df[col].median())
    df[col] = df[col].fillna(median_val)
    print(f"Kolumna '{col}': uzupełniono medianą -> {median_val}")

Kolumna 'Marital Status': uzupełniono modą -> 'Married'
Kolumna 'Gender': uzupełniono modą -> 'Male'
Kolumna 'Home Owner': uzupełniono modą -> 'Yes'
Kolumna 'Income': uzupełniono medianą -> 60000
Kolumna 'Children': uzupełniono medianą -> 2
Kolumna 'Cars': uzupełniono medianą -> 1
Kolumna 'Age': uzupełniono medianą -> 43


### Weryfikacja braków po imputacji
Porównujemy liczbę brakujących komórek w każdej kolumnie przed i po przeprowadzeniu procesu imputacji, aby upewnić się, że w zbiorze nie ma już żadnych braków danych.

In [ ]:
comparison_df = pd.DataFrame({
    'Przed imputacją': df_before.isnull().sum(),
    'Po imputacji': df.isnull().sum()
})
print("Tabela porównawcza liczby braków danych:")
print(comparison_df)

Tabela porównawcza liczby braków danych:
                  Przed imputacją  Po imputacji
ID                              0             0
Marital Status                  7             0
Gender                         11             0
Income                          6             0
Children                        8             0
Education                       0             0
Occupation                      0             0
Home Owner                      4             0
Cars                            9             0
Commute Distance                0             0
Region                          0             0
Age                             8             0
Purchased Bike                  0             0


### Zapisanie oczyszczonych danych do nowego pliku
Na zakończenie procesu zapisujemy oczyszczoną ramkę danych do nowego pliku o nazwie `data/sklep_rowerowy_oczyszczone.csv`. W ten sposób zachowujemy oryginalny plik bez zmian, a oczyszczone dane są gotowe do dalszej analizy.

In [ ]:
output_file = 'data/sklep_rowerowy_oczyszczone.csv'
df.to_csv(output_file, index=False)
print(f"Oczyszczone dane zostały pomyślnie zapisane do pliku: {output_file}")

Oczyszczone dane zostały pomyślnie zapisane do pliku: data/sklep_rowerowy_oczyszczone.csv


### Raport Skimpy

In [ ]:
from skimpy import skim
print("\n--- Wywołanie raportu strukturalnego skimpy ---")
skim(df)


--- Wywołanie raportu strukturalnego skimpy ---


╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 1000   │ │ string      │ 9     │                                                          │
│ │ Number of columns │ 13     │ │ int64       │ 4     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column      ┃ NA  ┃ NA %   ┃ mean    ┃ sd      ┃ p0      ┃ p25     ┃ p50     ┃ p75     ┃ p100    ┃ hist    ┃  │
│ ┡━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩  │
│ │ Income      │   0 │      0 │   56290 │   30980 │   10000 │   30000 │   60000 │   70000 │  170000 │  ▆▇▅▂▁  │  │
│ │ Children    │   0 │      0 │   1.911 │    1.62 │       0 │       0 │       2 │       3 │       5 │ ▇▅▆▃▃▂  │  │
│ │ Cars        │   0 │      0 │   1.451 │   1.118 │       0 │       1 │       1 │       2 │       4 │ ▆▆ ▇▂▁  │  │
│ │ Age         │   0 │      0 │   44.17 │   11.32 │      25 │      35 │      43 │      52 │      89 │  ▆▇▅▃▁  │  │
│ └─────────────┴─────┴────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓  │
│ ┃           ┃    ┃      ┃           ┃           ┃           ┃           ┃ chars per ┃ words per ┃ total      ┃  │
│ ┃ column    ┃ NA ┃ NA % ┃ shortest  ┃ longest   ┃ min       ┃ max       ┃ row       ┃ row       ┃ words      ┃  │
│ ┡━━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩  │
│ │ ID        │  0 │    0 │ 12496     │ 12496     │ 11000     │ 29447     │         5 │         1 │       1000 │  │
│ │ Marital   │  0 │    0 │ Single    │ Married   │ Married   │ Single    │      6.54 │         1 │       1000 │  │
│ │ Status    │    │      │           │           │           │           │           │           │            │  │
│ │ Gender    │  0 │    0 │ Male      │ Female    │ Female    │ Male      │      4.98 │         1 │       1000 │  │
│ │ Education │  0 │    0 │ Bachelors │ Partial   │ Bachelors │ Partial   │      12.8 │       1.8 │       1770 │  │
│ │           │    │      │           │ High      │           │ High      │           │           │            │  │
│ │           │    │      │           │ School    │           │ School    │           │           │            │  │
│ │ Occupatio │  0 │    0 │ Manual    │ Skilled   │ Clerical  │ Skilled   │      10.7 │       1.3 │       1255 │  │
│ │ n         │    │      │           │ Manual    │           │ Manual    │           │           │            │  │
│ │ Home      │  0 │    0 │ No        │ Yes       │ No        │ Yes       │      2.69 │         1 │       1000 │  │
│ │ Owner     │    │      │           │           │           │           │           │           │            │  │
│ │ Commute   │  0 │    0 │ 0-1 Miles │ 5-10      │ 0-1 Miles │ 5-10      │      9.19 │         2 │       2000 │  │
│ │ Distance  │    │      │           │ Miles     │     

Dla porządku wrzucamy też wygenerowany automatycznie raport skimpy. Dane mamy już wcześniej zweryfikowane i wyczyszczone, ale ten zestawienie fajnie i czytelnie podsumowuje całość w jednym miejscu. Do szczegółowej analizy średnich i rozkładów poszczególnych zmiennych wrócimy w dalszej części tego projektu.